# Praktikum Minggu 9: Konfigurasi Cloud via Emulator (Ministack / LocalStack)
Pastikan emulator sudah berjalan di `http://localhost:4566` sebelum mengeksekusi sel-sel di bawah ini.

## Install terlebih dahulu Ministack dan Stackport di Terminal komputer anda

#### Install dan Jalankan MiniStack
`pip install ministack && ministack`

#### Install dan Start StackPort
`pip install stackport && stackport`

Open http://localhost:8080

In [1]:
%pip install boto3

Note: you may need to restart the kernel to use updated packages.


In [2]:
import boto3

# Setup Endpoint
endpoint = 'http://localhost:4566' # bisa gunakan endpoint asli dari aws
region = 'us-east-1'
credentials = {
    'aws_access_key_id': '121212121212',
    'aws_secret_access_key': 'admin123'
}

# Inisialisasi EC2 dan S3 Client
ec2 = boto3.client('ec2', endpoint_url=endpoint, region_name=region, **credentials)
s3 = boto3.client('s3', endpoint_url=endpoint, region_name=region, **credentials)

print("Boto3 Clients berhasil diinisialisasi!")

Boto3 Clients berhasil diinisialisasi!


## Sesi Praktikum 1: Konfigurasi Jaringan (VPC & Subnet)
Membuat Virtual Private Cloud (VPC) dan Subnet publik.

In [7]:
# 1. Membuat VPC
vpc_response = ec2.create_vpc(CidrBlock='10.0.0.0/16')
vpc_id = vpc_response['Vpc']['VpcId']
print(f"VPC Berhasil Dibuat dengan ID: {vpc_id}")

# 2. Membuat Subnet Publik
subnet_response = ec2.create_subnet(
    VpcId=vpc_id,
    CidrBlock='127.0.0.1/24',
    AvailabilityZone='us-east-1'
)
public_subnet_id = subnet_response['Subnet']['SubnetId']
print(f"Subnet Publik Berhasil Dibuat dengan ID: {public_subnet_id}")

VPC Berhasil Dibuat dengan ID: vpc-86fea0c5036d14c10
Subnet Publik Berhasil Dibuat dengan ID: subnet-d3e7dc8d75c9f71ea


### Tugas Mandiri: Membuat Private Subnet
Tambahkan Subnet Private dengan rentang CIDR `10.0.2.0/24`.

In [ ]:
# Jawaban Tugas Mandiri: Membuat Private Subnet
private_subnet_response = ec2.create_subnet(
    VpcId=vpc_id,
    CidrBlock='127.0.0.1/24',
    AvailabilityZone='us-east-1'
)
private_subnet_id = private_subnet_response['Subnet']['SubnetId']
print(f"Subnet Privat Berhasil Dibuat dengan ID: {private_subnet_id}")

## Sesi Praktikum 2: Alokasi Compute Resource (EC2 Instance)
Melakukan provisioning VM/EC2 Instance ke dalam subnet publik yang telah dibuat.

In [4]:
# Eksekusi pembuatan VM Instance baru
instances = ec2.run_instances(
    ImageId='ami-mock-ubuntu', # ID AMI Dummy
    InstanceType='c8g.medium', # contoh : t2.micro
    MinCount=1,
    MaxCount=1,
    SubnetId=public_subnet_id
)

instance_id = instances['Instances'][0]['InstanceId']
state = instances['Instances'][0]['State']['Name']

print(f"Sukses Deploy VM! \nID Instansi: {instance_id} \nStatus saat ini: {state}")

Sukses Deploy VM! 
ID Instansi: i-4967d4a0b412a992e 
Status saat ini: running


## Sesi Praktikum 3: Manajemen Object Storage (Amazon S3)
Membuat bucket dan mengunggah object file teks ke dalamnya.

In [5]:
nama_bucket = 'ember-data-akademik-lokal'

# 1. Alokasi Namespace Bucket Baru
s3.create_bucket(Bucket=nama_bucket)
print(f"Bucket '{nama_bucket}' berhasil dibuat.")

# 2. Upload / Buat file ke dalam Object Storage
s3.put_object(
    Bucket=nama_bucket,
    Key='laporan_kuliah.txt',
    Body=b'Konten Hasil Kuliah Cloud Computing menggunakan Ministack.'
)
print("Unggah berkas 'laporan_kuliah.txt' berhasil diselesaikan.")

Bucket 'ember-data-akademik-lokal' berhasil dibuat.
Unggah berkas 'laporan_kuliah.txt' berhasil diselesaikan.


### Tugas Mandiri: Verifikasi Object S3
Tulis implementasi list objects untuk memastikan file sudah masuk ke bucket.

In [9]:
# Jawaban Tugas Mandiri: Menampilkan list object di S3
response = s3.list_objects_v2(Bucket=nama_bucket)

print(f"Daftar file di dalam bucket '{nama_bucket}':")
if 'Contents' in response:
    for obj in response['Contents']:
        print(f"- {obj['Key']} (Size: {obj['Size']} bytes)")
else:
    print("Bucket kosong.")

Daftar file di dalam bucket 'ember-data-akademik-lokal':
- laporan_kuliah.txt (Size: 58 bytes)


## Tugas Besar (Individu)
- Buatlah aplikasi fungsional dengan opsional UI (Desktop / Web / Notebook) untuk memanajemen S3 atau EC2
- Tidak boleh sama persis / mirip (80% UI mirip dianggap sama). Gunakan kreativitas anda
- Buat manual penggunaan aplikasi (pdf)
- Upload kodenya di github masing-masing (akan disediakan assigment di sinau)